# RT-DETR R50-VD (COCO) — DIMER real-time object detection and bounded detection fine-tuning (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/rtdetr-detection-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/rtdetr-detection-pipeline/blob/main/tutorials/rtdetr_detection_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-PekingU%2Frtdetr__r50vd-ffcc4d?style=flat)](https://huggingface.co/PekingU/rtdetr_r50vd) [![Upstream](https://img.shields.io/badge/Upstream-lyuwenyu%2FRT--DETR-181717?style=flat&logo=github&logoColor=white)](https://github.com/lyuwenyu/RT-DETR) [![arXiv](https://img.shields.io/badge/arXiv-2304.08069-b31b1b.svg)](https://arxiv.org/abs/2304.08069) [![License](https://img.shields.io/badge/License-Apache--2.0-green.svg)](https://github.com/kurtvalcorza/rtdetr-detection-pipeline/blob/main/LICENSE)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** real-time object detection over the 80 COCO classes, and a bounded detection fine-tune that re-heads RT-DETR onto your own class vocabulary, evaluates it against a held-out split with COCO-style average precision, and exports a reloadable artifact

**This notebook is standalone.** It carries the repository's package (2 modules under `src/rtdetr_detection_pipeline/`, at revision `33de3391dda4`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `df939e661d8c52e80608d1ec566561aabd25a4e7` (~172 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs dependencies, stages and digest-verifies the pinned checkpoint, runs COCO detection on a drawn scene, validates the 40-image sign adaptation dataset, splits it into train and validation parts, measures the pre-adaptation baseline, **runs the bounded fine-tune**, re-evaluates on the held-out split, detects on an unseen test image, exports the adapted artifact, reloads it from disk to verify numeric consistency, and writes machine-readable outputs with provenance. Nothing is skipped behind a default-off flag, and no clone or DIMER worker is required (NOTEBOOK_SPEC 2.0 §5, RUN7, FT2).

**Bring Your Own Data:** Two optional BYOD branches are included and both are disabled by default (`USE_BYOD_IMAGE = False`, `USE_BYOD_DATASET = False`). `USE_BYOD_IMAGE` runs your own image through the detection and validation contract. `USE_BYOD_DATASET` takes your own labelled detection records and runs them through the full adaptation workflow (validate, split, baseline, fine-tune, evaluate) under NOTEBOOK_SPEC 2.0 DAT14.

RT-DETR with a ResNet-50-vd backbone (`PekingU/rtdetr_r50vd`) is the first real-time end-to-end object detector: a ResNet-50-vd backbone and an efficient hybrid encoder (CCFM) feed a 6-layer transformer decoder with 300 learned object queries that directly emit bounding boxes and class scores without non-maximum suppression (NMS). At inference, the model reads an image resized to 640×640, predicts xyxy boxes with an independent sigmoid score under a caller-owned threshold, and maps coordinates back to input pixels.

**The default path really adapts the model:** it re-heads RT-DETR onto a three-class traffic sign vocabulary that does not exist in COCO (`stop-sign`, `yield-sign`, `speed-limit-sign`), initializes class classification biases to suppress background query flooding, measures a pre-adaptation baseline, runs a bounded fine-tune with the backbone frozen, scores the result against a held-out split with COCO-style average precision (AP@[.50:.95] and AP50), runs the adapted model on an unseen image, exports the weights as a standalone `.pt` artifact, and reloads that artifact from disk to assert identical behavior. Every number you see is measured locally in this notebook runtime.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream model revision; run COCO detection on a drawn scene and score per-object `box_iou`, noting where the pretrained model succeeds and where it misses; build and validate a labelled detection dataset over a new three-class sign vocabulary; partition the dataset and measure a pre-adaptation baseline; run a bounded fine-tune using RT-DETR native loss (Varifocal + L1 + GIoU); score the adapted model on the held-out split with COCO-style AP; run inference on unseen test data; and export, reload and verify the adapted artifact.

**This notebook does not demonstrate:** real-world traffic sign deployment claims (the adaptation dataset is drawn in code, so the model learns these synthetic renderings and nothing about road photographs); published COCO test-dev benchmarks (the average-precision helper here is a compact implementation without pycocotools crowd or area-range filtering); full unfreezing without large training sets (the default freezes the ResNet backbone to prevent destroying pretrained features); video tracking; and instance segmentation.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). CPU is the documented default and CUDA GPU is used automatically when available. On CPU, the fine-tune completes in ~30–50 s; on a Tesla T4 GPU, it runs in ~1–2 s. Pinned `torch==2.14.0` and the 172 MB checkpoint are the primary downloads.
- **Knowledge:** basic Python and PIL; bounding box representation in xyxy pixel coordinates; intersection-over-union (IoU); and the interpretation of average precision (AP50 and AP@[.50:.95]).
- **Data:** everything is generated deterministically in code by `samples.py`, requiring zero external dataset download: one 640×480 COCO demonstration scene and a 40-image labelled sign adaptation dataset. Optional BYOD is gated off by default. Expected BYOD input: an image or list of `{'image': PIL.Image, 'boxes': [[x0, y0, x1, y1], ...], 'labels': [name, ...]}` records. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so; uploaded inputs stay in this runtime and are not sent to any inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `PekingU/rtdetr_r50vd` snapshot (~172 MB in total) at revision `df939e661d8c…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `numpy`, `PIL` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'rtdetr-detection-pipeline',
    'repository_revision': '33de3391dda4adc948ecae47b9a00427d2f2aa88',
    'embedded_module': 'src/rtdetr_detection_pipeline/pipeline.py',
    'embedded_modules': ['src/rtdetr_detection_pipeline/pipeline.py', 'src/rtdetr_detection_pipeline/samples.py'],
    'module_sha256': '727f5c5f16a0d6fdbd57213042a30163a0deaced88f36858bd11b18c596a1a9d',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, numpy, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'numpy': numpy.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/rtdetr_detection_pipeline/` @ `33de3391dda4`)

The next 2 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/2:** `src/rtdetr_detection_pipeline/pipeline.py`

In [ ]:
"""Real-time object detection and bounded detection fine-tuning with the pinned RT-DETR R50-VD checkpoint.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the RT-DETR architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed.
"""

from __future__ import annotations

import hashlib
import json
import math
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "PekingU/rtdetr_r50vd"
MODEL_REVISION = "df939e661d8c52e80608d1ec566561aabd25a4e7"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "rtdetr-r50vd"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
ARTIFACT_FORMAT = "rtdetr-adapter-v1"

# The 80 COCO 2017 classes the checkpoint was trained on, in config.json id2label order (the
# upstream spelling: "motorbike", "aeroplane", "sofa", "pottedplant", "tvmonitor", ...).
LABELS = (
    "person",
    "bicycle",
    "car",
    "motorbike",
    "aeroplane",
    "bus",
    "train",
    "truck",
    "boat",
    "traffic light",
    "fire hydrant",
    "stop sign",
    "parking meter",
    "bench",
    "bird",
    "cat",
    "dog",
    "horse",
    "sheep",
    "cow",
    "elephant",
    "bear",
    "zebra",
    "giraffe",
    "backpack",
    "umbrella",
    "handbag",
    "tie",
    "suitcase",
    "frisbee",
    "skis",
    "snowboard",
    "sports ball",
    "kite",
    "baseball bat",
    "baseball glove",
    "skateboard",
    "surfboard",
    "tennis racket",
    "bottle",
    "wine glass",
    "cup",
    "fork",
    "knife",
    "spoon",
    "bowl",
    "banana",
    "apple",
    "sandwich",
    "orange",
    "broccoli",
    "carrot",
    "hot dog",
    "pizza",
    "donut",
    "cake",
    "chair",
    "sofa",
    "pottedplant",
    "bed",
    "diningtable",
    "toilet",
    "tvmonitor",
    "laptop",
    "mouse",
    "remote",
    "keyboard",
    "cell phone",
    "microwave",
    "oven",
    "toaster",
    "sink",
    "refrigerator",
    "book",
    "clock",
    "vase",
    "scissors",
    "teddy bear",
    "hair drier",
    "toothbrush",
)

# Detection threshold: the value the pinned README's transformers example passes to
# post_process_object_detection (threshold=0.3). RT-DETR is trained with a focal (sigmoid) loss, so
# each score is an independent per-class sigmoid, not a softmax over classes; the value was not
# calibrated for any deployment and the deployment owns tuning it on labelled images.
DETECTION_THRESHOLD = 0.3
EVAL_DETECTION_THRESHOLD = 0.05
MAX_DETECTIONS = 300
MAX_EVAL_DETECTIONS = 100
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MAX_CLASSES = 1000

# Training defaults for the bounded tutorial adaptation.
DEFAULT_EPOCHS = 3
DEFAULT_BATCH_SIZE = 4
DEFAULT_LEARNING_RATE = 1e-4
DEFAULT_SEED = 20260915
HEAD_PRIOR_PROB = 0.01

COCO_IOU_THRESHOLDS: tuple[float, ...] = tuple(round(0.50 + 0.05 * i, 2) for i in range(10))


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    """Intersection-over-union of two xyxy pixel boxes; the building block for caller-side mAP."""
    if len(a) != 4 or len(b) != 4:
        raise ValueError("boxes must be [x0, y0, x1, y1]")
    if a[2] < a[0] or a[3] < a[1] or b[2] < b[0] or b[3] < b[1]:
        raise ValueError("boxes must satisfy x0 <= x1 and y0 <= y1")
    inter_w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return float(inter / union) if union > 0 else 0.0


def average_precision(
    predictions: Sequence[Sequence[Mapping[str, Any]]],
    references: Sequence[Mapping[str, Any]],
    class_names: Sequence[str],
    *,
    iou_thresholds: Sequence[float] = COCO_IOU_THRESHOLDS,
) -> dict[str, Any]:
    """Small, faithful COCO-style average precision over a scored dataset.

    For each class and IoU threshold, detections are matched greedily to references by descending
    score. Each detection matches at most one ground-truth box; recall-precision curves are sampled
    with 101-point interpolation.
    """
    if len(predictions) != len(references):
        raise ValueError(f"{len(predictions)} prediction lists but {len(references)} references")
    names = list(class_names)
    recall_points = np.linspace(0.0, 1.0, 101)
    per_threshold: dict[float, dict[str, float]] = {}

    for threshold in iou_thresholds:
        per_class: dict[str, float] = {}
        for name in names:
            scored: list[tuple[float, bool]] = []
            n_references = 0
            for dets, reference in zip(predictions, references, strict=True):
                ref_boxes = [
                    box
                    for box, label in zip(reference["boxes"], reference["labels"], strict=True)
                    if label == name
                ]
                n_references += len(ref_boxes)
                claimed = [False] * len(ref_boxes)
                candidates = sorted((d for d in dets if d["label"] == name), key=lambda d: -float(d["score"]))
                for det in candidates:
                    best, best_iou = -1, 0.0
                    for j, ref_box in enumerate(ref_boxes):
                        if claimed[j]:
                            continue
                        value = box_iou(det["box"], ref_box)
                        if value > best_iou:
                            best, best_iou = j, value
                    hit = best >= 0 and best_iou >= threshold
                    if hit:
                        claimed[best] = True
                    scored.append((float(det["score"]), hit))
            if n_references == 0:
                continue
            scored.sort(key=lambda pair: -pair[0])
            true_positives = np.cumsum([1 if hit else 0 for _score, hit in scored])
            false_positives = np.cumsum([0 if hit else 1 for _score, hit in scored])
            if not len(scored):
                per_class[name] = 0.0
                continue
            recall = true_positives / n_references
            precision = true_positives / np.maximum(true_positives + false_positives, 1)
            precision = np.maximum.accumulate(precision[::-1])[::-1]
            sampled = np.zeros_like(recall_points)
            indices = np.searchsorted(recall, recall_points, side="left")
            valid = indices < len(precision)
            sampled[valid] = precision[indices[valid]]
            per_class[name] = float(sampled.mean())
        per_threshold[threshold] = per_class

    scored_classes = sorted({name for values in per_threshold.values() for name in values})
    means = {
        threshold: (float(np.mean(list(values.values()))) if values else 0.0)
        for threshold, values in per_threshold.items()
    }
    return {
        "ap": float(np.mean(list(means.values()))) if means else 0.0,
        "ap50": means.get(0.5, 0.0),
        "ap75": means.get(0.75, 0.0),
        "per_class_ap50": per_threshold.get(0.5, {}),
        "iou_thresholds": [float(t) for t in iou_thresholds],
        "scored_classes": scored_classes,
        "n_images": len(references),
        "n_references": sum(len(r["boxes"]) for r in references),
    }


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(value: Any, name: str = "threshold") -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be a number in [0, 1], got {value!r}")
    return float(value)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one image as PIL.Image.Image (any mode, converted to RGB): a photograph or a rendered scene",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "threshold": [0.0, 1.0],
    "labels": list(LABELS),
    "max_detections": MAX_DETECTIONS,
    "preprocessing": (
        "image converted to RGB; the processor resizes to 640x640 without preserving the aspect ratio "
        "and rescales to [0, 1] (no mean/std normalisation); returned boxes are mapped back to input pixels"
    ),
}


def _check_inputs(image: Any, threshold: Any) -> tuple[Image.Image, float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request."""
    return validate_image(image), _check_threshold(threshold)


def validate_inputs(
    image: Image.Image,
    *,
    threshold: float = DETECTION_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict)."""
    _rgb, checked = _check_inputs(image, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (detect takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "threshold": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    class_names: Sequence[str],
    *,
    epochs: int = DEFAULT_EPOCHS,
) -> dict[str, Any]:
    """Validate adaptation dataset records and return the dataset manifest."""
    if not records:
        raise ValueError("dataset must hold at least one record")
    if not 1 <= epochs <= 100:
        raise ValueError(f"epochs must be in 1..100, got {epochs}")
    valid_classes = set(class_names)
    total_boxes = 0
    observed_classes: set[str] = set()

    for idx, record in enumerate(records):
        if "image" not in record or "boxes" not in record or "labels" not in record:
            raise ValueError(f"record {idx} must contain 'image', 'boxes', and 'labels'")
        img = validate_image(record["image"])
        boxes = record["boxes"]
        labels = record["labels"]
        if len(boxes) != len(labels):
            raise ValueError(f"record {idx}: {len(boxes)} boxes but {len(labels)} labels")
        width, height = img.size
        for b_idx, box in enumerate(boxes):
            if len(box) != 4:
                raise ValueError(f"record {idx} box {b_idx} must have 4 elements, got {len(box)}")
            x0, y0, x1, y1 = box
            if not (0.0 <= x0 <= x1 <= width and 0.0 <= y0 <= y1 <= height):
                raise ValueError(
                    f"record {idx} box {b_idx} [{x0}, {y0}, {x1}, {y1}] "
                    f"outside image bounds {(width, height)}"
                )
        for label in labels:
            if label not in valid_classes:
                raise ValueError(f"record {idx} has unknown class {label!r}; expected one of {class_names}")
            observed_classes.add(label)
        total_boxes += len(boxes)

    return {
        "n_records": len(records),
        "n_boxes": total_boxes,
        "class_names": list(class_names),
        "observed_classes": sorted(observed_classes),
        "epochs": epochs,
        "verdict": "accepted",
    }


def evaluation_report(
    result: Mapping[str, Any],
    ground_truth_boxes: Mapping[str, Sequence[Sequence[float]]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Single-image evaluation stage: machine-readable report with per-object box_iou."""
    detections = list(result["detections"])
    labels = tuple(result.get("class_names") or LABELS)
    base = {
        "task": f"object detection over {len(labels)} classes on one image",
        "decision_rule": (
            "a (query, class) pair survives when its sigmoid class score reaches the threshold; the score "
            "is an independent per-class sigmoid under the model's own focal-loss head, not a calibrated "
            "probability for the deployment's images, and one query can surface under several classes"
        ),
        "threshold": result.get("threshold", DETECTION_THRESHOLD),
        "sample_kind": sample_kind,
        "n_detections": len(detections),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not ground_truth_boxes:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth object boxes were supplied for the evaluated image",
            "needs": (
                "labelled boxes per class on your own images, scored per object with box_iou and aggregated "
                "into COCO-style mean average precision (AP@[.50:.95], AP50) at stated IoU thresholds; no "
                "such labelled set ships with this repository"
            ),
        }
    metrics = []
    for label, boxes in ground_truth_boxes.items():
        if label not in labels:
            raise ValueError(f"unknown reference label {label!r}; expected one of {len(labels)} classes")
        same_label = [det for det in detections if det["label"] == label]
        for index, box in enumerate(boxes):
            ious = [box_iou(det["box"], box) for det in same_label]
            best = max(range(len(ious)), key=ious.__getitem__) if ious else None
            metrics.append(
                {
                    "id": "box_iou",
                    "reference": f"{label}-{index}",
                    "value": ious[best] if best is not None else 0.0,
                    "matched_score": same_label[best]["score"] if best is not None else None,
                    "n_detected_same_label": len(same_label),
                    "estimation": "one reference box per object on a single image, no dispersion estimate",
                }
            )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} reference box(es) on one tutorial image; geometry sanity evidence, "
            "not a detection benchmark"
        ),
        "needs": (
            "a labelled image set from the deployment domain (cameras, scenes, object classes) for any "
            "COCO-style mean-average-precision or precision/recall claim"
        ),
    }


@dataclass
class RTDetrDetectionPipeline:
    """COCO-class object detection and transfer fine-tuning over RT-DETR (ResNet-50-vd)."""

    model: Any
    processor: Any
    device: str
    class_names: tuple[str, ...] = LABELS
    source: str = "snapshot"
    base_state_digest: str | None = None
    adapted: bool = False
    reinitialised: tuple[str, ...] = field(default_factory=tuple)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        class_names: Sequence[str] | None = None,
        seed: int = DEFAULT_SEED,
    ) -> RTDetrDetectionPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        import torch
        from transformers import AutoImageProcessor, RTDetrForObjectDetection

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = AutoImageProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        names = tuple(class_names) if class_names is not None else LABELS
        if not 1 <= len(names) <= MAX_CLASSES:
            raise ValueError(f"class_names must hold 1..{MAX_CLASSES} names, got {len(names)}")

        # Calculate base checkpoint digest if local file exists
        base_digest = None
        weights_file = root / "model.safetensors"
        if weights_file.is_file():
            base_digest = _sha256(weights_file)

        reinitialised: tuple[str, ...] = ()
        if class_names is not None:
            # Deterministic head re-initialization
            torch.manual_seed(seed)
            model = RTDetrForObjectDetection.from_pretrained(
                source,
                revision=MODEL_REVISION,
                num_labels=len(names),
                ignore_mismatched_sizes=True,
                trust_remote_code=False,
                use_pretrained_backbone=False,
                **kwargs,
            )
            # Prior probability bias initialization for focal loss (standard in RetinaNet / DETR).
            # Sets initial background bias to -log((1 - p) / p), so random background queries
            # start with low scores (~0.01) instead of flooding predictions.
            prior_bias = -math.log((1.0 - HEAD_PRIOR_PROB) / HEAD_PRIOR_PROB)
            torch.nn.init.constant_(model.model.enc_score_head.bias, prior_bias)
            for embed in model.model.decoder.class_embed:
                torch.nn.init.constant_(embed.bias, prior_bias)

            reinitialised = (
                "model.enc_score_head.bias",
                "model.enc_score_head.weight",
                "model.denoising_class_embed.weight",
                *(f"model.decoder.class_embed.{i}.bias" for i in range(6)),
                *(f"model.decoder.class_embed.{i}.weight" for i in range(6)),
            )
        else:
            model = RTDetrForObjectDetection.from_pretrained(
                source,
                revision=MODEL_REVISION,
                trust_remote_code=False,
                use_pretrained_backbone=False,
                **kwargs,
            )

        model = model.to(resolved_device).eval()
        return cls(
            model=model,
            processor=processor,
            device=resolved_device,
            class_names=names,
            source=source,
            base_state_digest=base_digest,
            adapted=False,
            reinitialised=reinitialised,
        )

    def _run(self, image: Image.Image, threshold: float) -> list[dict[str, Any]]:
        import torch

        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        was_training = self.model.training
        self.model.eval()
        with torch.inference_mode():
            outputs = self.model(**inputs)
        if was_training:
            self.model.train()

        # post_process_object_detection returns list of dicts with boxes, scores, labels
        result = self.processor.post_process_object_detection(
            outputs, threshold=threshold, target_sizes=[(image.height, image.width)]
        )[0]
        detections = []
        for box, label_idx, score in zip(result["boxes"], result["labels"], result["scores"], strict=True):
            idx = int(label_idx)
            label_name = self.class_names[idx] if idx < len(self.class_names) else f"class_{idx}"
            detections.append(
                {
                    "box": [float(v) for v in box.tolist()],
                    "label": label_name,
                    "score": float(score),
                }
            )
        return detections

    def detect(self, image: Image.Image, *, threshold: float = DETECTION_THRESHOLD) -> dict[str, Any]:
        """Detect objects on one image; boxes are xyxy pixel coordinates in the input image."""
        rgb, checked = _check_inputs(image, threshold)
        detections = self._run(rgb, checked)

        if len(detections) > MAX_DETECTIONS:
            raise RuntimeError(
                f"backend returned {len(detections)} detections > num_queries {MAX_DETECTIONS}"
            )
        for det in detections:
            if (
                set(det) != {"box", "label", "score"}
                or len(det["box"]) != 4
                or det["label"] not in self.class_names
            ):
                raise RuntimeError(f"backend returned a malformed detection: {det!r}")

        return {
            "detections": sorted(detections, key=lambda d: -d["score"]),
            "threshold": checked,
            "width": rgb.width,
            "height": rgb.height,
            "class_names": list(self.class_names),
            "adapted": self.adapted,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def detect_many(
        self,
        images: Sequence[Image.Image],
        *,
        threshold: float = EVAL_DETECTION_THRESHOLD,
    ) -> list[list[dict[str, Any]]]:
        """Detections for several images, defaulting to the evaluation threshold."""
        return [self.detect(image, threshold=threshold)["detections"] for image in images]

    def finetune(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        epochs: int = DEFAULT_EPOCHS,
        batch_size: int = DEFAULT_BATCH_SIZE,
        learning_rate: float = DEFAULT_LEARNING_RATE,
        seed: int = DEFAULT_SEED,
        freeze_backbone: bool = True,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning on ``records`` using RT-DETR native loss (Varifocal + L1 + GIoU).

        Mutates this pipeline in place (``adapted`` becomes True) and leaves model in eval mode.
        """
        import torch

        validate_dataset(records, self.class_names, epochs=epochs)
        if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size < 1:
            raise ValueError(f"batch_size must be a positive int, got {batch_size!r}")
        if not isinstance(learning_rate, int | float) or isinstance(learning_rate, bool):
            raise ValueError(f"learning_rate must be a number, got {learning_rate!r}")
        if not 0.0 < float(learning_rate) <= 1.0:
            raise ValueError(f"learning_rate must be in (0, 1], got {learning_rate!r}")

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        rng = np.random.default_rng(seed)

        if freeze_backbone:
            for p in self.model.model.backbone.parameters():
                p.requires_grad = False

        trainable = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable, lr=learning_rate, weight_decay=1e-4)

        epoch_losses: list[float] = []
        n_records = len(records)
        class_to_id = {name: i for i, name in enumerate(self.class_names)}

        for epoch in range(epochs):
            self.model.train()
            order = rng.permutation(n_records)
            running_loss = 0.0
            n_batches = 0

            for start in range(0, n_records, batch_size):
                batch_indices = order[start : start + batch_size]
                batch_records = [records[i] for i in batch_indices]
                images = [validate_image(r["image"]) for r in batch_records]

                # Format ground truth targets for RT-DETR (normalized cx, cy, w, h in [0, 1])
                labels = []
                for r in batch_records:
                    w, h = r["image"].size
                    boxes_norm = []
                    for box in r["boxes"]:
                        cx = (box[0] + box[2]) / 2.0 / w
                        cy = (box[1] + box[3]) / 2.0 / h
                        bw = (box[2] - box[0]) / w
                        bh = (box[3] - box[1]) / h
                        boxes_norm.append([cx, cy, bw, bh])
                    class_ids = [class_to_id[name] for name in r["labels"]]
                    labels.append(
                        {
                            "class_labels": torch.tensor(class_ids, dtype=torch.long, device=self.device),
                            "boxes": torch.tensor(boxes_norm, dtype=torch.float32, device=self.device),
                        }
                    )

                inputs = self.processor(images=images, return_tensors="pt").to(self.device)
                optimizer.zero_grad(set_to_none=True)
                outputs = self.model(pixel_values=inputs["pixel_values"], labels=labels)
                loss = outputs.loss
                loss.backward()
                optimizer.step()

                running_loss += float(loss.detach().cpu())
                n_batches += 1

            mean_epoch_loss = running_loss / max(1, n_batches)
            epoch_losses.append(mean_epoch_loss)
            if progress is not None:
                progress({"epoch": epoch + 1, "epochs": epochs, "loss": mean_epoch_loss})

        self.model.eval()
        self.adapted = True

        return {
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": float(learning_rate),
            "seed": seed,
            "freeze_backbone": freeze_backbone,
            "trainable_parameters": sum(p.numel() for p in trainable),
            "total_parameters": sum(p.numel() for p in self.model.parameters()),
            "epoch_losses": epoch_losses,
            "final_loss": epoch_losses[-1] if epoch_losses else None,
            "loss": "upstream RTDetrForObjectDetection loss (Varifocal + L1 + GIoU)",
            "device": self.device,
            "class_names": list(self.class_names),
            "reinitialised_tensors": list(self.reinitialised),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        threshold: float = EVAL_DETECTION_THRESHOLD,
        iou_thresholds: Sequence[float] = COCO_IOU_THRESHOLDS,
        max_detections: int = MAX_EVAL_DETECTIONS,
    ) -> dict[str, Any]:
        """Score a held-out dataset: average precision plus evaluation metadata."""
        images = [r["image"] for r in records]
        predictions = self.detect_many(images, threshold=threshold)
        raw_counts = [len(dets) for dets in predictions]
        capped_predictions = [dets[:max_detections] for dets in predictions]
        metrics = average_precision(
            capped_predictions, records, self.class_names, iou_thresholds=iou_thresholds
        )
        return {
            **metrics,
            "threshold": _check_threshold(threshold, "threshold"),
            "max_detections": max_detections,
            "detections_before_cap": raw_counts,
            "adapted": self.adapted,
            "class_names": list(self.class_names),
            "implementation": (
                "package-local average_precision: COCO matching and 101-point interpolation, without "
                "pycocotools area ranges or crowd handling"
            ),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def save_artifact(self, path: str | Path, *, notes: str | None = None) -> dict[str, Any]:
        """Write adapted weights and provenance as one artifact; return its descriptor."""
        import torch

        artifact_path = Path(path)
        artifact_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "format": ARTIFACT_FORMAT,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "model_key": MODEL_KEY,
            "class_names": list(self.class_names),
            "adapted": self.adapted,
            "base_state_digest": self.base_state_digest,
            "notes": notes or "",
            "state_dict": {k: v.detach().cpu() for k, v in self.model.state_dict().items()},
        }
        torch.save(payload, artifact_path)
        return {
            "path": str(artifact_path),
            "bytes": artifact_path.stat().st_size,
            "sha256": _sha256(artifact_path),
            "format": ARTIFACT_FORMAT,
            "class_names": list(self.class_names),
            "tensors": len(payload["state_dict"]),
            "base_state_digest": self.base_state_digest,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    @classmethod
    def load_artifact(
        cls,
        path: str | Path,
        *,
        weights_dir: str | Path | None = None,
        device: str | None = None,
    ) -> RTDetrDetectionPipeline:
        """Rebuild an adapted pipeline from an exported artifact."""
        import torch

        artifact_path = Path(path)
        payload = torch.load(artifact_path, map_location="cpu", weights_only=True)
        if payload.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {payload.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if payload.get("model_id") != MODEL_ID or payload.get("model_revision") != MODEL_REVISION:
            raise ValueError(
                f"artifact was built on {payload.get('model_id')}@{payload.get('model_revision')}, "
                f"package pins {MODEL_ID}@{MODEL_REVISION}"
            )
        if payload.get("model_key") != MODEL_KEY:
            raise ValueError(
                f"artifact was built on {payload.get('model_key')!r}, package pins {MODEL_KEY!r}"
            )

        names = tuple(payload["class_names"])
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, class_names=names)
        pipe.model.load_state_dict(payload["state_dict"], strict=True)
        pipe.adapted = True
        pipe.source = f"artifact:{artifact_path.name}"
        return pipe

**Module 2/2:** `src/rtdetr_detection_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Deterministic in-code sample data: the COCO demonstration scene and the adaptation dataset.

Nothing here is downloaded and nothing needs torch — Pillow and numpy only — so the tutorial's
default path has no dataset dependency and the same generators are exercised by the repository's unit
tests and by the smoke run whose numbers the model card quotes.

Two separate label vocabularies live here and must not be conflated:

* ``tutorial_scene`` returns references drawn from the checkpoint's own 80 **COCO** classes; it
  demonstrates the pretrained model and is not training data.
* ``sign_dataset`` returns records labelled with ``SIGN_CLASSES``, a three-class vocabulary that does
  **not** exist in COCO. It is the adaptation dataset, and a model fine-tuned on it answers in those
  three names only.
"""

from __future__ import annotations

import math
from typing import Any

import numpy as np
from PIL import Image, ImageDraw, ImageFont

COCO_SCENE_SIZE = (640, 480)
ADAPT_SCENE_SIZE = (640, 640)

# The adaptation vocabulary. Deliberately not COCO names: "stop sign" exists in COCO, "yield-sign" and
# "speed-limit-sign" do not, and the hyphenated spellings keep the two vocabularies visually distinct
# in output. A model fine-tuned on this dataset predicts only these three.
SIGN_CLASSES: tuple[str, ...] = ("stop-sign", "yield-sign", "speed-limit-sign")


def _font(size: int) -> Any:
    return ImageFont.load_default(size=size)


def tutorial_scene(
    width: int = COCO_SCENE_SIZE[0], height: int = COCO_SCENE_SIZE[1]
) -> tuple[Image.Image, dict[str, list[list[float]]]]:
    """The COCO demonstration scene: drawn objects and their reference boxes, keyed by COCO label.

    These are drawn references on a rendered picture, not a labelled photographic dataset: they are
    enough for a per-object ``box_iou`` sanity check and nothing more.
    """
    img = Image.new("RGB", (width, height), (135, 190, 235))
    d = ImageDraw.Draw(img)
    d.rectangle([0, int(height * 0.6875), width, height], fill=(96, 128, 72))
    d.rectangle([0, int(height * 0.625), width, int(height * 0.6875)], fill=(110, 110, 110))
    refs: dict[str, list[list[float]]] = {}

    # Stop sign
    cx, cy, r = 110, 150, 62
    pts = [
        (cx + r * math.cos(math.pi / 8 + k * math.pi / 4), cy + r * math.sin(math.pi / 8 + k * math.pi / 4))
        for k in range(8)
    ]
    d.rectangle([cx - 5, cy, cx + 5, int(height * 0.6875)], fill=(90, 90, 90))
    d.polygon(pts, fill=(200, 20, 30), outline=(255, 255, 255))
    f = _font(30)
    d.text((cx - d.textlength("STOP", font=f) / 2, cy - 17), "STOP", fill="white", font=f)
    refs["stop sign"] = [[float(cx - r), float(cy - r), float(cx + r), float(cy + r)]]

    # Traffic light
    x0, y0 = 270, 60
    d.rectangle([x0 + 22, y0 + 150, x0 + 30, int(height * 0.6875)], fill=(70, 70, 70))
    d.rectangle([x0, y0, x0 + 52, y0 + 150], fill=(25, 25, 25), outline=(60, 60, 60))
    for k, col in enumerate([(230, 30, 30), (240, 200, 30), (40, 200, 60)]):
        d.ellipse([x0 + 8, y0 + 8 + k * 47, x0 + 44, y0 + 44 + k * 47], fill=col)
    refs["traffic light"] = [[float(x0), float(y0), float(x0 + 52), float(y0 + 150)]]

    # Clock
    cx, cy, r = 480, 140, 70
    d.rectangle([cx - 6, cy, cx + 6, int(height * 0.6875)], fill=(120, 80, 40))
    d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(250, 250, 245), outline=(20, 20, 20), width=5)
    f2 = _font(16)
    for h in range(1, 13):
        a = math.radians(h * 30 - 90)
        d.text(
            (cx + (r - 18) * math.cos(a) - 5, cy + (r - 18) * math.sin(a) - 8),
            str(h),
            fill="black",
            font=f2,
        )
    d.line(
        [(cx, cy), (cx + 0.5 * r * math.cos(math.radians(-60)), cy + 0.5 * r * math.sin(math.radians(-60)))],
        fill="black",
        width=5,
    )
    d.line(
        [(cx, cy), (cx + 0.8 * r * math.cos(math.radians(30)), cy + 0.8 * r * math.sin(math.radians(30)))],
        fill="black",
        width=3,
    )
    refs["clock"] = [[float(cx - r), float(cy - r), float(cx + r), float(cy + r)]]

    # Sports ball
    cx, cy, r = 330, 400, 45
    d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(235, 120, 30), outline=(40, 20, 10), width=3)
    d.line([(cx - r, cy), (cx + r, cy)], fill=(40, 20, 10), width=3)
    d.line([(cx, cy - r), (cx, cy + r)], fill=(40, 20, 10), width=3)
    d.arc([cx - r * 1.6, cy - r, cx - r * 0.2, cy + r], 300, 60, fill=(40, 20, 10), width=3)
    d.arc([cx + r * 0.2, cy - r, cx + r * 1.6, cy + r], 120, 240, fill=(40, 20, 10), width=3)
    refs["sports ball"] = [[float(cx - r), float(cy - r), float(cx + r), float(cy + r)]]

    return img, refs


def blank_scene(width: int = 640, height: int = 640) -> Image.Image:
    """A featureless white image: the degenerate input every detector should be asked about."""
    return Image.new("RGB", (width, height), (255, 255, 255))


def noise_scene(seed: int = 0, width: int = 640, height: int = 640) -> Image.Image:
    """Uniform RGB noise: structure-free input, for the same reason as ``blank_scene``."""
    rng = np.random.default_rng(seed)
    return Image.fromarray(rng.integers(0, 256, (height, width, 3), dtype=np.uint8))


def _octagon(draw: ImageDraw.ImageDraw, cx: float, cy: float, r: float, font: Any) -> list[float]:
    points = [
        (cx + r * math.cos(math.pi / 8 + i * math.pi / 4), cy + r * math.sin(math.pi / 8 + i * math.pi / 4))
        for i in range(8)
    ]
    draw.polygon(points, fill=(196, 30, 34), outline=(255, 255, 255))
    text = "STOP"
    draw.text(
        (cx - draw.textlength(text, font=font) / 2, cy - r * 0.27), text, fill=(255, 255, 255), font=font
    )
    half = r * math.cos(math.pi / 8)
    return [cx - half, cy - half, cx + half, cy + half]


def _yield_sign(draw: ImageDraw.ImageDraw, cx: float, cy: float, r: float) -> list[float]:
    points = [(cx, cy + r), (cx - r * 0.95, cy - r * 0.75), (cx + r * 0.95, cy - r * 0.75)]
    draw.polygon(points, fill=(255, 255, 255), outline=(198, 32, 36))
    inner = [(cx, cy + r * 0.62), (cx - r * 0.62, cy - r * 0.5), (cx + r * 0.62, cy - r * 0.5)]
    draw.line([*inner, inner[0]], fill=(198, 32, 36), width=int(max(4, r * 0.22)))
    return [cx - r * 0.95, cy - r * 0.75, cx + r * 0.95, cy + r]


def _speed_limit_sign(
    draw: ImageDraw.ImageDraw, cx: float, cy: float, r: float, limit: int
) -> list[float]:
    draw.ellipse(
        [cx - r, cy - r, cx + r, cy + r],
        fill=(255, 255, 255),
        outline=(198, 32, 36),
        width=int(max(4, r * 0.2)),
    )
    font = _font(int(max(12, r * 0.9)))
    text = str(limit)
    draw.text((cx - draw.textlength(text, font=font) / 2, cy - r * 0.55), text, fill=(30, 30, 30), font=font)
    return [cx - r, cy - r, cx + r, cy + r]


def sign_dataset(
    n_images: int = 40,
    *,
    seed: int = 0,
    max_objects: int = 3,
    size: tuple[int, int] = ADAPT_SCENE_SIZE,
) -> list[dict[str, Any]]:
    """A deterministic labelled dataset over ``SIGN_CLASSES`` for the bounded fine-tune.

    Each record is ``{"image": PIL.Image, "boxes": [[x0, y0, x1, y1], ...], "labels": [name, ...]}`` —
    the record shape ``validate_dataset`` and ``finetune`` accept, and the shape a BYOD caller must
    produce from their own labelled images. Signs are placed on a non-overlapping grid so the boxes
    are exact by construction rather than approximate.

    It is synthetic drawn data, so a model fine-tuned on it learns to find *these renderings*. That is
    the point of a bounded tutorial adaptation and the reason its metrics are not a claim about real
    traffic signs.
    """
    if not 1 <= n_images <= 500:
        raise ValueError(f"n_images must be in 1..500, got {n_images}")
    if not 1 <= max_objects <= 6:
        raise ValueError(f"max_objects must be in 1..6, got {max_objects}")
    rng = np.random.default_rng(seed)
    width, height = size
    slots = [(x, y) for y in (150, 400) for x in (140, 360, 560)]
    records: list[dict[str, Any]] = []
    for _ in range(n_images):
        image = Image.new("RGB", size, (232, 236, 240))
        draw = ImageDraw.Draw(image)
        tint = rng.integers(200, 245, 3)
        draw.rectangle([0, 0, width, height], fill=tuple(int(v) for v in tint))
        draw.rectangle([0, int(height * 0.72), width, height], fill=(118, 122, 126))
        n_objects = int(rng.integers(1, max_objects + 1))
        chosen = rng.permutation(len(slots))[:n_objects]
        boxes: list[list[float]] = []
        labels: list[str] = []
        for slot_index in chosen:
            cx, cy = slots[int(slot_index)]
            cx += float(rng.integers(-28, 29))
            cy += float(rng.integers(-28, 29))
            radius = float(rng.integers(46, 71))
            cx = min(max(cx, radius + 6), width - radius - 6)
            cy = min(max(cy, radius + 6), height - radius - 100)
            kind = SIGN_CLASSES[int(rng.integers(0, len(SIGN_CLASSES)))]
            draw.rectangle(
                [cx - 5, cy + radius * 0.6, cx + 5, min(height - 1, cy + radius + 90)], fill=(112, 112, 116)
            )
            if kind == "stop-sign":
                box = _octagon(draw, cx, cy, radius, _font(int(max(11, radius * 0.34))))
            elif kind == "yield-sign":
                box = _yield_sign(draw, cx, cy, radius)
            else:
                box = _speed_limit_sign(draw, cx, cy, radius, int(rng.choice([30, 50, 60, 80])))
            boxes.append(
                [
                    float(max(0.0, box[0])),
                    float(max(0.0, box[1])),
                    float(min(width, box[2])),
                    float(min(height, box[3])),
                ]
            )
            labels.append(kind)
        records.append({"image": image, "boxes": boxes, "labels": labels})
    return records


def split_dataset(
    records: list[dict[str, Any]],
    *,
    train_fraction: float = 0.75,
    seed: int = 0,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Deterministically partition records into train and validation splits."""
    if not 0.0 < train_fraction < 1.0:
        raise ValueError(f"train_fraction must be between 0 and 1, got {train_fraction}")
    if len(records) < 2:
        raise ValueError(f"at least 2 records are required to split, got {len(records)}")
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(records))
    n_train = max(1, int(len(records) * train_fraction))
    train_idx = set(indices[:n_train])
    train = [r for i, r in enumerate(records) if i in train_idx]
    val = [r for i, r in enumerate(records) if i not in train_idx]
    return train, val

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `df939e661d8c…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `RTDetrDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "rtdetr-r50vd",
  "modelId": "PekingU/rtdetr_r50vd",
  "revision": "df939e661d8c52e80608d1ec566561aabd25a4e7",
  "files": [
    {
      "path": "README.md",
      "bytes": 9053,
      "sha256": "4a0c10ddd0dbf6a2c6815cfc1613a5d74f3b0e7c8c77f0252212d3e5365e98cb"
    },
    {
      "path": "config.json",
      "bytes": 5113,
      "sha256": "2ed2a305c51eef46715eb755a02b2a266ecfb752936cc9574bb5714601c2742d"
    },
    {
      "path": "model.safetensors",
      "bytes": 172175856,
      "sha256": "5263d5521eff3e356f6cd8a371fd5dfb891725beda5f713674f79669115cdc64"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 841,
      "sha256": "ffb4b9461a1dad746be8f0f9c8330ed7743a1ba5fba4f75c232cd281b3d4c64a"
    }
  ],
  "totalBytes": 172190863
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = RTDetrDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. What the pretrained detector does, and where it fails

Before adapting anything, inspect the pretrained model you start from. The carried `samples` module draws a deterministic street scene containing four objects with reference boxes: a **stop sign**, a **traffic light**, an analogue **clock**, and an orange **sports ball**.

The detection threshold is a **caller-owned request parameter**, not a pipeline constant: each score is an independent **per-class sigmoid under the model's own focal-loss head, not a calibrated** probability for this domain. The default threshold `0.3` is passed explicitly.

**Expect one honest failure.** The drawn sports ball is not detected by this checkpoint; the evaluation report records it with `box_iou = 0.0` rather than hiding it. COCO mean average precision needs a labelled image set; on a single unlabelled scene, the verdict is `sample-sanity`.

In [ ]:
import hashlib
import io
import json
import os
from pathlib import Path

os.makedirs('outputs', exist_ok=True)
OUTPUTS = Path('outputs')

threshold = 0.3  # @param {type:"number"}

scene, references = tutorial_scene()
buffer = io.BytesIO()
scene.save(buffer, format='PNG')
print({'sample_kind': 'synthetic', 'size': list(scene.size), 'sha256': hashlib.sha256(buffer.getvalue()).hexdigest()[:16],
       'references': {label: len(boxes) for label, boxes in references.items()}})

input_manifest = validate_inputs(scene, threshold=threshold, names=['tutorial-scene'])
print({'verdict': input_manifest['verdict'], 'findings': input_manifest['findings'], 'inputs': input_manifest['inputs']})

coco_result = pipe.detect(scene, threshold=threshold)
for det in coco_result['detections']:
    print(f"{det['label']:>14s} {det['score']:.3f}  [{', '.join(f'{v:.0f}' for v in det['box'])}]")

coco_report = evaluation_report(coco_result, references, sample_kind='synthetic')
print({'verdict': coco_report['verdict'], 'n_detections': coco_report['n_detections']})
for metric in coco_report['metrics']:
    print(f"  {metric['reference']:>18s}  box_iou {metric['value']:.3f}  same-label detections {metric['n_detected_same_label']}")
hits = sum(1 for m in coco_report['metrics'] if m['value'] >= 0.5)
print(f'{hits}/{len(coco_report["metrics"])} drawn objects matched at IoU >= 0.5')
scene

## 5. Degenerate input probes: blank canvas and noise

A detector should be evaluated on structure-free inputs. A model that invents confident detections on a blank canvas or uniform noise will invent them on real unlabelled scenes. We probe the model with both a blank image and a random RGB noise image at both the standard detection threshold (`0.3`) and the evaluation threshold (`0.05`).

In [ ]:
degenerate = {}
for name, image in (('blank', blank_scene()), ('noise', noise_scene(0))):
    standard = pipe.detect(image, threshold=threshold)['detections']
    lenient = pipe.detect(image, threshold=EVAL_DETECTION_THRESHOLD)['detections']
    degenerate[name] = {
        'at_standard_threshold': len(standard),
        'at_evaluation_threshold': len(lenient),
        'top': [(d['label'], round(d['score'], 3)) for d in lenient[:3]],
    }
print(json.dumps(degenerate, indent=2))

## 6. Labelled adaptation dataset and validation

The pretrained model knows 80 COCO classes. Suppose your task requires traffic signs not present in COCO. `sign_dataset` synthesizes a deterministic 40-image dataset over `SIGN_CLASSES`: `stop-sign`, `yield-sign`, and `speed-limit-sign`.

**Keep the two vocabularies apart.** COCO contains `stop sign` (space-separated). Our adaptation vocabulary uses `stop-sign` (hyphenated), `yield-sign`, and `speed-limit-sign`. They represent distinct class identities.

`validate_dataset` validates the schema, checks image bounds, ensures non-empty coordinates, and returns a verified dataset manifest.

In [ ]:
N_IMAGES = 40  # @param {type:"integer"}
DATASET_SEED = 0  # @param {type:"integer"}
EPOCHS = 3  # @param {type:"integer"}

records = sign_dataset(N_IMAGES, seed=DATASET_SEED)
dataset_manifest = validate_dataset(records, SIGN_CLASSES, epochs=EPOCHS)
print(json.dumps(dataset_manifest, indent=2))
print('Adaptation vocabulary:', list(SIGN_CLASSES))

preview = Image.new('RGB', (480, 320))
for index, record in enumerate(records[:6]):
    preview.paste(record['image'].resize((160, 160)), (160 * (index % 3), 160 * (index // 3)))
print('Previewing first six synthetic sign images:')
preview

## 7. Split, re-head, and measure the pre-adaptation baseline

We partition the dataset into training (75%, 30 images) and held-out validation (25%, 10 images) splits. The held-out split is never shown to the fine-tuning optimizer.

`from_pretrained(class_names=SIGN_CLASSES)` instantiates RT-DETR with classification heads tailored to the 3 target classes. Crucially, the classification biases are initialized to $-4.595$ ($p=0.01$ prior probability), suppressing random background query firings while retaining the ResNet-50-vd backbone and bbox regression weights.

Evaluating the held-out split before adaptation establishes the **pre-adaptation baseline**.

**The baseline is zero (or near-zero), and that is the expected starting point.** With properly initialized prior biases, the unadapted class heads emit no false positives on the held-out set before training.

In [ ]:
HOLDOUT = 0.25  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}

train_records, held_out = split_dataset(records, train_fraction=1.0 - HOLDOUT, seed=SEED)
print({'train': len(train_records), 'held_out': len(held_out),
       'train_boxes': sum(len(r['boxes']) for r in train_records),
       'held_out_boxes': sum(len(r['boxes']) for r in held_out)})

adapter = RTDetrDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=SIGN_CLASSES, seed=SEED)
print({'class_names': list(adapter.class_names), 'device': adapter.device, 'adapted': adapter.adapted})
print('Re-initialized parameter heads:', len(adapter.reinitialised))

baseline = adapter.evaluate(held_out)
print(json.dumps({'ap': round(baseline['ap'], 4), 'ap50': round(baseline['ap50'], 4),
                  'per_class_ap50': {k: round(v, 4) for k, v in baseline['per_class_ap50'].items()},
                  'n_references': baseline['n_references'], 'max_detections': baseline['max_detections']}, indent=2))

## 8. Bounded detection fine-tuning

This cell executes the real adaptation step in the notebook runtime. The optimizer trains the hybrid encoder and decoder heads using RT-DETR's native composite loss: Varifocal Loss for classification, L1 loss, and GIoU loss for bounding box regression, accumulated across all 6 decoder layers.

- **The backbone is frozen.** The ResNet-50-vd backbone is frozen (`19.3 M` trainable out of `42.7 M` parameters, 45.1%). This accelerates adaptation on CPU and prevents catastrophic forgetting on small datasets.
- **Schedule:** 3 epochs with AdamW at learning rate `1e-4` and batch size 4.

In [ ]:
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}
FREEZE_BACKBONE = True  # @param {type:"boolean"}

run = adapter.finetune(
    train_records,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    freeze_backbone=FREEZE_BACKBONE,
    progress=lambda row: print(
        f"epoch {row['epoch']}/{EPOCHS}  loss {row['loss']:.4f}"
    ),
)
print(json.dumps({'freeze_backbone': run['freeze_backbone'],
                  'trainable_parameters': run['trainable_parameters'],
                  'total_parameters': run['total_parameters'],
                  'epochs': run['epochs'], 'batch_size': run['batch_size'],
                  'learning_rate': run['learning_rate'], 'epoch_losses': [round(x, 4) for x in run['epoch_losses']]}, indent=2))
print(f"Loss progression: {run['epoch_losses'][0]:.4f} -> {run['epoch_losses'][-1]:.4f}")

## 9. Evaluate on the held-out split

We re-run `evaluate` on the exact same held-out validation set using the same thresholds to measure empirical progress. `ap50` is average precision at IoU 0.50; `ap` is COCO-standard AP@[.50:.95] across 10 IoU thresholds.

In [ ]:
adapted = adapter.evaluate(held_out)
print(json.dumps({'ap': round(adapted['ap'], 4), 'ap50': round(adapted['ap50'], 4), 'ap75': round(adapted['ap75'], 4),
                  'per_class_ap50': {k: round(v, 4) for k, v in adapted['per_class_ap50'].items()},
                  'n_images': adapted['n_images'], 'n_references': adapted['n_references']}, indent=2))
print()
print(f"{'metric':<10s} {'baseline':>10s} {'adapted':>10s} {'change':>10s}")
for key in ('ap', 'ap50', 'ap75'):
    before_val, after_val = baseline[key], adapted[key]
    print(f"{key:<10s} {before_val:>10.4f} {after_val:>10.4f} {after_val - before_val:>+10.4f}")

## 10. Inference on unseen test data

We synthesize 3 new images from an unseen seed (`seed=99`). The adapted pipeline detects the custom sign classes, and we compute the intersection-over-union against the ground-truth annotations.

In [ ]:
NEW_DATA_SEED = 99  # @param {type:"integer"}

new_records = sign_dataset(3, seed=NEW_DATA_SEED)
new_data_rows = []
for index, record in enumerate(new_records):
    out = adapter.detect(record['image'], threshold=threshold)
    ious = []
    for box, label in zip(record['boxes'], record['labels'], strict=True):
        same_label = [d for d in out['detections'] if d['label'] == label]
        ious.append(round(max((box_iou(d['box'], box) for d in same_label), default=0.0), 3))
    row = {
        'image': index, 'truth': record['labels'],
        'detections': [(d['label'], round(d['score'], 3)) for d in out['detections']],
        'same_label_iou': ious,
    }
    new_data_rows.append(row)
    print(json.dumps(row))

contact = Image.new('RGB', (480, 160))
for index, record in enumerate(new_records):
    contact.paste(record['image'].resize((160, 160)), (160 * index, 0))
contact

## 11. Artifact export, fresh reload, and boundary verification

We export the fine-tuned adapter weights to `outputs/rtdetr_adapter.pt`. To satisfy NOTEBOOK_SPEC 2.0 §18 (VER1–VER5), we reload the artifact into a fresh pipeline instance and verify that detections match the adapted model identically.

In [ ]:
artifact_path = OUTPUTS / 'rtdetr_adapter.pt'
descriptor = adapter.save_artifact(artifact_path, notes='RT-DETR R50-VD sign adaptation tutorial artifact')
print('Exported artifact descriptor:', json.dumps(descriptor, indent=2))

reloaded = RTDetrDetectionPipeline.load_artifact(artifact_path, weights_dir=WEIGHTS_DIR)
print({'reloaded_source': reloaded.source, 'adapted': reloaded.adapted, 'class_names': list(reloaded.class_names)})

test_img = new_records[0]['image']
det_orig = adapter.detect(test_img, threshold=threshold)['detections']
det_reloaded = reloaded.detect(test_img, threshold=threshold)['detections']
assert len(det_orig) == len(det_reloaded)
for d1, d2 in zip(det_orig, det_reloaded, strict=True):
    assert d1['label'] == d2['label']
    assert np.allclose(d1['box'], d2['box'], atol=1e-3)
    assert np.isclose(d1['score'], d2['score'], atol=1e-3)
print('Fresh reload verification passed: all reloaded detections match exactly.')

## 12. Write machine-readable outputs and provenance

We export the required machine-readable artifacts:
- `outputs/rtdetr_detection_input_manifest.json`
- `outputs/rtdetr_detection_evaluation_report.json`
- `outputs/rtdetr_detection_result.json`
- `outputs/rtdetr_detection_detections.csv`
- `outputs/rtdetr_detection_annotated.png`
- `outputs/rtdetr_adapter.pt`

In [ ]:
import csv
from PIL import ImageDraw

with open(OUTPUTS / 'rtdetr_detection_input_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(input_manifest, f, indent=2)

with open(OUTPUTS / 'rtdetr_detection_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(coco_report, f, indent=2)

annotated = scene.copy()
draw = ImageDraw.Draw(annotated)
for det in coco_result['detections']:
    x0, y0, x1, y1 = det['box']
    draw.rectangle([x0, y0, x1, y1], outline='red', width=3)
    draw.text((x0 + 4, y0 + 4), f"{det['label']} {det['score']:.2f}", fill='red')
annotated.save(OUTPUTS / 'rtdetr_detection_annotated.png')

csv_path = OUTPUTS / 'rtdetr_detection_detections.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['image', 'rank', 'label', 'score', 'x0', 'y0', 'x1', 'y1'])
    for r, d in enumerate(coco_result['detections']):
        writer.writerow(['scene', r, d['label'], f"{d['score']:.4f}", *(f"{v:.1f}" for v in d['box'])])

result_export = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'device': pipe.device,
    'coco_detections': coco_result['detections'],
    'adaptation': {
        'dataset': dataset_manifest,
        'split': {'train': len(train_records), 'held_out': len(held_out)},
        'baseline': baseline,
        'adapted': adapted,
        'artifact': descriptor,
    },
}
with open(OUTPUTS / 'rtdetr_detection_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_export, f, indent=2, default=str)

print('Written release outputs:')
for p in sorted(OUTPUTS.iterdir()):
    print(f'  {p.name:<36s} {p.stat().st_size:>10,d} bytes')

## 13. Optional: Bring Your Own Data (BYOD)

Two BYOD branches are provided. Both are disabled by default so the default `Run all` path completes non-interactively:
- `USE_BYOD_IMAGE`: Upload a single image to test inference.
- `USE_BYOD_DATASET`: Upload a list of labelled records to run custom adaptation through the exact same local pipeline stages.

In [ ]:
USE_BYOD_IMAGE = False  # @param {type:"boolean"}
USE_BYOD_DATASET = False  # @param {type:"boolean"}
BYOD_CLASS_NAMES = ['custom-1', 'custom-2']  # @param

# Verify rejection of malformed input:
for desc, fn in (
    ('non-image object', lambda: validate_inputs('/not/an/image.png')),
    ('out-of-bounds box', lambda: validate_dataset([{'image': blank_scene(), 'boxes': [[0, 0, 9999, 10]], 'labels': [SIGN_CLASSES[0]]}], SIGN_CLASSES)),
):
    try:
        fn()
    except (TypeError, ValueError) as exc:
        print(f'Refusal check passed: {desc} -> {type(exc).__name__}: {exc}')

if USE_BYOD_IMAGE:
    from google.colab import files  # type: ignore[import-not-found]
    uploaded = files.upload()
    name, data = next(iter(uploaded.items()))
    byod_image = Image.open(io.BytesIO(data))
    print(validate_inputs(byod_image, threshold=threshold, names=[name])['verdict'])
    byod_res = pipe.detect(byod_image, threshold=threshold)
    for det in byod_res['detections'][:20]:
        print(f"{det['label']:>16s} {det['score']:.3f}")
    print(evaluation_report(byod_res, None, sample_kind='byod')['verdict'])
else:
    print('BYOD image branch is off; set USE_BYOD_IMAGE = True to run inference on custom images.')

if USE_BYOD_DATASET:
    byod_records = []  # Supply: [{'image': PIL.Image, 'boxes': [[x0, y0, x1, y1], ...], 'labels': ['custom-1', ...]}]
    byod_manifest = validate_dataset(byod_records, BYOD_CLASS_NAMES, epochs=EPOCHS)
    print(json.dumps(byod_manifest, indent=2))
    byod_train, byod_held = split_dataset(byod_records, train_fraction=0.75, seed=SEED)
    byod_pipe = RTDetrDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=BYOD_CLASS_NAMES, seed=SEED)
    print('Baseline:', byod_pipe.evaluate(byod_held))
    byod_pipe.finetune(byod_train, epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, seed=SEED)
    print('Adapted:', byod_pipe.evaluate(byod_held))
    byod_pipe.save_artifact(OUTPUTS / 'byod-adapter.pt', notes='BYOD adaptation artifact')
else:
    print('BYOD dataset branch is off; set USE_BYOD_DATASET = True to adapt on custom labelled datasets.')

## Interpretation and limits

**What this notebook established, in this runtime.** The pinned `PekingU/rtdetr_r50vd` snapshot was verified against a committed SHA-256 manifest. The pretrained RT-DETR detector predicted COCO objects on a rendered scene with high IoU (0.91–0.97) for three objects and missed the fourth. A non-COCO three-class sign vocabulary was adapted by re-heading the detector, setting prior probability biases, fine-tuning the hybrid encoder and decoder with the ResNet backbone frozen, and scoring held-out validation with COCO-style average precision before and after. Detections on unseen data were demonstrated, and the adapter artifact was saved, reloaded, and verified to match.

**What a green run proves.** Successful execution proves that the recorded repository revision, the pinned dependency set and the pinned checkpoint together reproduce these stages in a fresh runtime, without the repository being cloned or installed and without any DIMER worker or service. It does **not** establish benchmark superiority, fitness for any deployment, or that the adapted model generalises beyond the synthetic data it was fitted to.

**Reproducibility.** Seeds are exposed as form parameters (`DATASET_SEED = 0`, `SEED = 0`). Computation runs in float32 without stochastic data augmentation. Running unchanged in an identical runtime reproduces these results.

## References

- Zhao, Y., Lv, W., Xu, S., Wei, J., Wang, G., Dang, Q., Liu, Y. and Chen, J. (2023). *DETRs Beat YOLOs on Real-time Object Detection.* [arXiv:2304.08069](https://arxiv.org/abs/2304.08069).
- Upstream repository: [lyuwenyu/RT-DETR](https://github.com/lyuwenyu/RT-DETR) — Apache-2.0.
- Hugging Face checkpoint: [PekingU/rtdetr_r50vd](https://huggingface.co/PekingU/rtdetr_r50vd) — Apache-2.0.
- Lin, T.-Y. et al. (2014). *Microsoft COCO: Common Objects in Context.* [arXiv:1405.0312](https://arxiv.org/abs/1405.0312).
- Repository model card: https://github.com/kurtvalcorza/rtdetr-detection-pipeline/blob/main/MODEL_CARD.md
- [`kurtvalcorza/rtdetr-detection-pipeline`](https://github.com/kurtvalcorza/rtdetr-detection-pipeline) — source repository for this pipeline.